In [6]:
import pandas as pd
import numpy as np

# ── Chargement ──────────────────────────────────────────────────
df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2021.csv', sep=';')
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2022.csv', sep=';')
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2023.csv', sep=';')
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2024.csv', sep=',')

# Ajouter la colonne année avant de merger (important pour les analyses)
df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

# ── Merge (même ordre que ton collègue) ─────────────────────────
usagers_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)
print(usagers_df.head())

        Num_Acc      id_usager    id_vehicule num_veh  place  catu  grav  \
0  202400000001  203  988  581  155  781  758     A01      1     1     3   
1  202400000001  203  988  582  155  781  759     B01      1     1     1   
2  202400000002  203  988  579  155  781  757     A01     10     3     3   
3  202400000002  203  988  580  155  781  757     A01      1     1     1   
4  202400000003  203  988  574  155  781  756     A01      2     2     4   

   sexe  an_nais  trajet  secu1  secu2  secu3  locp actp  etatp  annee  
0     1   2003.0       2      1     -1     -1    -1   -1     -1   2024  
1     1   1997.0       4      1     -1     -1    -1   -1     -1   2024  
2     2   1927.0       5      0     -1     -1     3    3      1   2024  
3     1   1987.0       4      1      0     -1     3    3      1   2024  
4     2   2007.0       5      8      0     -1    -1   -1     -1   2024  


In [7]:
# -1 = "non renseigné" dans le fichier BAAC → on remplace par NaN
cols_sentinel = ['place', 'grav', 'sexe', 'trajet',
                 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']

usagers_df[cols_sentinel] = usagers_df[cols_sentinel].replace(-1, np.nan)

print("✅ Valeurs -1 remplacées par NaN")
print(usagers_df.isna().sum())

✅ Valeurs -1 remplacées par NaN
Num_Acc             0
id_usager           0
id_vehicule         0
num_veh             0
place              30
catu                0
grav              419
sexe            10632
an_nais         11118
trajet          11269
secu1            9865
secu2          223921
secu3          489711
locp           240813
actp                0
etatp          467713
annee               0
dtype: int64


In [8]:
# Source : documentation officielle ONISR
usagers_df['grav'] = usagers_df['grav'].map({
    1: 'Indemne',
    2: 'Tué',
    3: 'Blessé hospitalisé',
    4: 'Blessé léger'
})

usagers_df['catu'] = usagers_df['catu'].map({
    1: 'Conducteur',
    2: 'Passager',
    3: 'Piéton'
})

usagers_df['sexe'] = usagers_df['sexe'].map({
    1: 'Masculin',
    2: 'Féminin'
})

usagers_df['trajet'] = usagers_df['trajet'].map({
    0: 'Non renseigné',
    1: 'Domicile – travail',
    2: 'Domicile – école',
    3: 'Courses – achats',
    4: 'Utilisation professionnelle',
    5: 'Promenade – loisirs',
    9: 'Autre'
})

secu_map = {
    0: 'Aucun équipement',
    1: 'Ceinture',
    2: 'Casque',
    3: 'Dispositif enfants',
    4: 'Gilet réfléchissant',
    5: 'Airbag (2RM/3RM)',
    6: 'Gants (2RM/3RM)',
    7: 'Gants + Airbag (2RM/3RM)',
    8: 'Non déterminable',
    9: 'Autre'
}
usagers_df['secu1'] = usagers_df['secu1'].map(secu_map)
usagers_df['secu2'] = usagers_df['secu2'].map(secu_map)
usagers_df['secu3'] = usagers_df['secu3'].map(secu_map)

print("✅ Codes remplacés par leurs labels")

✅ Codes remplacés par leurs labels


In [9]:
print(f"Shape final : {usagers_df.shape}")

print("\n--- Gravité ---")
print(usagers_df['grav'].value_counts(dropna=False))

print("\n--- Catégorie usager ---")
print(usagers_df['catu'].value_counts(dropna=False))

print("\n--- Sexe ---")
print(usagers_df['sexe'].value_counts(dropna=False))

print("\n--- NaN par colonne ---")
print(usagers_df.isna().sum())

Shape final : (506886, 17)

--- Gravité ---
grav
Indemne               215092
Blessé léger          201026
Blessé hospitalisé     76750
Tué                    13599
NaN                      419
Name: count, dtype: int64

--- Catégorie usager ---
catu
Conducteur    377740
Passager       91141
Piéton         38005
Name: count, dtype: int64

--- Sexe ---
sexe
Masculin    338907
Féminin     157347
NaN          10632
Name: count, dtype: int64

--- NaN par colonne ---
Num_Acc             0
id_usager           0
id_vehicule         0
num_veh             0
place              30
catu                0
grav              419
sexe            10632
an_nais         11118
trajet          11269
secu1            9865
secu2          223921
secu3          489711
locp           240813
actp                0
etatp          467713
annee               0
dtype: int64


In [10]:
usagers_df.to_csv('Dataset/usagers_2021_2024_clean.csv', index=False, sep=';')
print(f"✅ Fichier exporté : {len(usagers_df):,} lignes x {usagers_df.shape[1]} colonnes")

✅ Fichier exporté : 506,886 lignes x 17 colonnes
